<a href="https://colab.research.google.com/github/marchionesss/Yandex_practicum_projects/blob/main/startups_predicting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Описание проекта

Предоставлены данные о стартапах, функционировавших в период с 1970 по 2018 годы.

**Цель проекта:**
🔸На оновании представленных данных тренировочной и тестовой выборки нужно создать модель, позволяющую определить какие стартапы жизнеспособны.
🔸Провести полноценный разведочный анализ и сформировать рекомендации будущим создателям стартапов (какие факторы влияют на успешность стартапа).

**Ход исследования:**

Инициализация необходимых инструментов.

Загрузка и подготовка данных:  загрузка датасетов, предобработка данных: изучение общей информации, выявление пропущенных значений, дубликатов и других аномалий.

Анализ данных: изучение структуры данных, распределений, визуализация частоты встречаемости данных, выявление основных тенденций, визуализация зависимостей, описание их типов, проведение корреляционного анализа.

Обучение моделей: использование пайплайнов для подбора гиперпараметров. Обучение модели, которая предскажет вероятность закрытия стартапа.

In [3]:
pip install ydata-profiling

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.1/400.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.8/687.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 53.4 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=2f1d729499970c25c163ac7620d671be90a0f0eb013ef8d0b6609ebaa06fb353
  Stored in directory: /root/.cache/pip/wheels/8d/55/1a/19cd535375ed1ede0c996405ebffe34b196d78e2d9545723a2
Successfully built htmlmin


In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:.0f}'.format

from ydata_profiling import ProfileReport

import matplotlib

import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
from numpy import sqrt

import seaborn as sns

#import plotly.express as px

import sklearn
from sklearn.model_selection import train_test_split
#from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
#(
#    MinMaxScaler,
#    RobustScaler,
#    StandardScaler)

from sklearn.linear_model import LinearRegression, Ridge

from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

## Загрузка и подготовка данных

In [9]:
data_0 = pd.read_csv('kaggle_startups_train_28062024.csv')
data_1 = pd.read_csv('kaggle_startups_test_28062024.csv')
data_2 = pd.read_csv('kaggle_startups_sample_submit_28062024.csv')

In [12]:
data_0.head()

,name,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at,closed_at
0,Lunchgate,Online Reservations|Restaurants,828626,operating,CHE,25,Zurich,Zürich,2,2009-10-17,2011-05-01,2014-12-01,NaN
1,EarLens,Manufacturing|Medical|Medical Devices,42935019,operating,USA,CA,SF Bay Area,Redwood City,4,2005-01-01,2010-05-04,2014-02-25,NaN
2,Reviva Pharmaceuticals,Biotechnology,35456381,operating,USA,CA,SF Bay Area,San Jose,3,2006-01-01,2012-08-20,2014-07-02,NaN
3,Sancilio and Company,Health Care,22250000,operating,NaN,NaN,NaN,NaN,3,2004-01-01,2011-09-01,2014-07-18,NaN
4,WireTough Cylinders,Manufacturing,NaN,operating,USA,VA,VA - Other,Bristol,1,2010-05-12,2012-02-01,2012-02-01,NaN


In [13]:
data_1.head()

,name,category_list,funding_total_usd,country_code,state_code,region,city,funding_rounds,first_funding_at,last_funding_at,lifetime
0,Crystalsol,Clean Technology,2819200,NIC,17,NaN,NaN,1,2009-07-01,2009-07-01,3501
1,JBI Fish & Wings,Hospitality,NaN,USA,TN,TN - Other,Humboldt,1,2010-07-28,2010-07-28,2717
2,COINPLUS,Finance,428257,LUX,3,Esch-sur-alzette,Esch-sur-alzette,2,2014-05-15,2014-09-18,1295
3,Imagine Communications,Software|Video|Video Streaming,34700000,USA,CA,San Diego,San Diego,4,2005-01-01,2010-04-20,4748
4,DNA13,Software,4530000,CAN,ON,Ottawa,Ottawa,1,2007-05-08,2007-05-08,6209


In [19]:
data_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13125 entries, 0 to 13124
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               13125 non-null  object 
 1   category_list      12534 non-null  object 
 2   funding_total_usd  10547 non-null  float64
 3   country_code       11743 non-null  object 
 4   state_code         11430 non-null  object 
 5   region             11536 non-null  object 
 6   city               11538 non-null  object 
 7   funding_rounds     13125 non-null  int64  
 8   first_funding_at   13125 non-null  object 
 9   last_funding_at    13125 non-null  object 
 10  lifetime           13125 non-null  int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.1+ MB


In [14]:
data_2.head()

,name,status
0,Crystalsol,closed
1,JBI Fish & Wings,operating
2,COINPLUS,closed
3,Imagine Communications,closed
4,DNA13,operating


In [18]:
data_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13125 entries, 0 to 13124
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    13125 non-null  object
 1   status  13125 non-null  object
dtypes: object(2)
memory usage: 205.2+ KB


`Прочитаны файлы для работы. Данные в таблицах соответствуют описанию, структура данных файлов тренировочной и тестовой выборки не идентична. Все загрузки отобразились корректно. Тренировочная выборка содержит целевой признак status. Для обзора и исследовательского анализа данных буду использовать инструмент ydata-profiling`

In [15]:
data_0_pro = ProfileReport(data_0, title="Profiling Report")

In [16]:
#в файл
data_0_pro.to_file("data_0_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 13/13 [00:01<00:00,  6.52it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

В тестовой выборке 52516 наблюдений. Есть пропуски - 12.5%. Явных дубликатов нет. Пройдем по столбцам, их 13.

1 name: текстовая переменная. одно значение отсутствует(не имеет значения). Наименования уникальные.

2 category_list: текстовая переменная. 4.7% значений отсутствует. Всего 20140 уникальных категорий. Причем одна ячейка может содержать несколько категорий через разделитель. Можно попробовать группировку по ключевым словам. Наиболее частые категории: програмное обеспчение, биотехнологии, здоровье.

3 funding_total_usd: числовая переменная. Общая сумма финансирования в USD. 19.2% значений отсутствует. Есть аномалии и выбросы (от 1 до 50, от 4 630 000 000 ). Среднее сильно отличается от медианы. Аномалии необходимо удалить (значение более 8 207 450 000). Возможно потребуется категоризация.

4 status: категориальная, целевая. (закрыт или действующий) Пропусков нет. Дисбаланс: 90% действующих.

5 country_code: текстовый. 10.5%  значений отсутствует. Определяется 22 уникальные категории, используется латиница и кириллица. Необходимо унифицировать. 63.2% - это США. есть значения, которые явно не относятся к кодам стран.

6 state_code: код штата. текстовый. 12.9% значений отсутствует. 44 уникальные категории, есть коды, есть длинные слова и цифры. Чаще всего са. есть значения, которые явно не относятся к кодам штата.

7 region: текст. 12.1% пропусков. 220 уникальных регионов. кириллица и латиница. заливы и площади. Кажется этот столбец имеет небольшую важность из-за большой вариативности.

8 city: текст. 12.1% пропусков. 2304 уникальных городов. кириллица и латиница.

9 funding_rounds: Количество раундов финансирования. Числа. без пропусков. от 1 до 19. Среднее 1,7. Медиана 1. Один раунд у 63.8%. Можно попробовать категоризацию.

10 founded_at: Дата основания. дата. без пропусков. Основные наблюдения за 2008-2014 годы.

11 first_funding_at: Дата первого раунда финансирования. дата. без пропусков. Основные наблюдения за 2008-2014 годы.

12

Проверить на логику.

Представляется, что залог успеха состоит в скурпулезном соотношении данных в столбцах 5-8. Если данные отсутствуют в этих столбцах, то строки можно удалить






